# Roboflow → YOLOv8: 학습 & 체크포인트 이어서 학습 (Colab 실습)

**Roboflow**에서 이미 **Train/Valid/Test 분할**이 끝난 데이터셋을 가져와서,
Ultralytics **YOLOv8**로 학습하고, **중단 후 체크포인트에서 이어서 학습(resume)** 하는 방법을 정리한 실습용 템플릿입니다.

## 구성
1. 환경 준비 (GPU 확인, 라이브러리 설치)
2. Roboflow 데이터셋 다운로드 (Export 링크 or SDK)
3. YOLOv8 새로 학습 시작
4. 체크포인트에서 이어서 학습 (resume)
5. 학습 결과 확인 & 추론 테스트

In [ ]:
import os

# 1. 마운트 해제 (혹시 이전에 마운트된 경우)
# drive.flush_and_unmount() # 이 함수는 Drive가 마운트된 상태에서만 작동함

# 2. 문제가 되는 폴더를 강제로 비우고 다시 생성
!rm -rf /content/drive/*
!rm -rf /content/drive # 폴더 자체를 삭제 후
os.makedirs('/content/drive', exist_ok=True) # 다시 생성

# 3. Drive 마운트 재시도
from google.colab import drive
drive.mount('/content/drive')

rm: cannot remove '/content/drive/MyDrive': Operation canceled
rm: cannot remove '/content/drive/MyDrive': Operation canceled
rm: cannot remove '/content/drive/.shortcut-targets-by-id': Operation canceled
rm: cannot remove '/content/drive/.Trash-0': Directory not empty
rm: cannot remove '/content/drive/.Encrypted/MyDrive': Operation canceled
rm: cannot remove '/content/drive/.Encrypted/.shortcut-targets-by-id': Operation canceled
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0. 런타임 준비: GPU 확인 (Colab)
- Colab 상단: *런타임 → 런타임 유형 변경 → 하드웨어 가속기: GPU* 로 설정
- 아래 코드를 실행해 GPU가 잡히는지 확인하세요.

In [ ]:
import torch, platform
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ GPU가 연결되지 않았습니다. Colab 런타임에서 GPU를 켜세요.")

Python: 3.12.12
PyTorch: 2.8.0+cu126
CUDA available: True
GPU: Tesla T4


## 1. 라이브러리 설치
- Ultralytics YOLOv8 (학습/추론)
- Roboflow SDK (선택: API로 데이터 다운로드 시)
- 기타 유틸

In [ ]:
!pip -q install ultralytics roboflow==1.* -U
from ultralytics import YOLO
import os, glob, shutil, json
print("Ultralytics 모듈 로드 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 47.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 모듈 로드 완료


## 2. Roboflow 데이터셋 다운로드

### 방법 A) Export 링크로 직접 다운로드 (권장, 가장 간단)
1) Roboflow 프로젝트 → **Download Dataset** → **YOLOv8** 선택
2) 생성된 **Download Code**에서 **`curl -L`** 다운로드 링크를 복사
3) 아래 `DATASET_URL`에 붙여넣고 실행

※ 이 방식은 **이미 분할된 데이터셋**을 그대로 가져옵니다.

In [ ]:
DATASET_URL = "https://app.roboflow.com/ds/xyAERW3GoI?key=heVeMRUEeY"
DATA_DIR = "/content/datasets"
os.makedirs(DATA_DIR, exist_ok=True)

if DATASET_URL.startswith("http"):
    !curl -L "$DATASET_URL" -o /content/dataset.zip
    !unzip -o /content/dataset.zip -d "$DATA_DIR" > /dev/null
    !find "$DATA_DIR" -maxdepth 3 -name data.yaml -print
else:
    print(" DATASET_URL을 Roboflow Export 링크로 바꿔주세요.")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   903  100   903    0     0   2070      0 --:--:-- --:--:-- --:--:--  2071
100 22.3M  100 22.3M    0     0  5218k      0  0:00:04  0:00:04 --:--:-- 8345k
/content/datasets/data.yaml


## 3. YOLOv8 새로 학습 시작
- `model`: 사전학습 가중치 선택 (예: `yolov8n.pt`)
- `data`: 위에서 받은 `data.yaml` 경로
- `epochs`, `imgsz`, `batch`: 수업 환경에 맞게 조정
- `project`, `name`: 결과가 저장될 폴더명

체크포인트(`best.pt`, `last.pt`)는 `runs/detect/<name>/weights/`에 자동 저장됩니다.

**3-2어그멘테이션 적용**

**어그멘테이션 3-3**

In [ ]:
from ultralytics import YOLO
import glob, os

yaml_candidates = glob.glob(os.path.join("/content/datasets", "**", "data.yaml"), recursive=True)
assert len(yaml_candidates) > 0, "data.yaml을 찾지 못했습니다. 다운로드 경로를 확인하세요."
DATA_YAML = yaml_candidates[0]
print("사용할 data.yaml:", DATA_YAML)

# 1. 모델 로드: yolov8n.pt (사전 학습된 가중치로 처음부터 시작)
model = YOLO("yolov8n.pt")

# 2. 새로운 학습 시작
results = model.train(
    data=DATA_YAML,
    epochs=60,
    imgsz=640,
    batch=16,
    project="runs_detect",
    name="final",

    # 학습률(Learning Rate) 조정
    lr0=0.005,          # 초기 학습률 조정 (기본값 0.01보다 낮춤)
    lrf=0.001,          # 최종 학습률 조정 (기본값 0.01보다 낮춰 미세 조정)

    # 데이터 증강 강화 (과적합 방지)
    degrees=10.0,
    perspective=0.0003,
    shear=3.0,
    flipud=0.5,
    mixup=0.1,

    # 규제 강화 (과적합 방지)
    weight_decay=0.001,
)

사용할 data.yaml: /content/datasets/data.yaml
Ultralytics 8.3.225 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/data.yaml, degrees=10.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.001, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=final4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pat

## 5. 학습 결과 확인 & 추론 테스트
- 결과 폴더의 `results.png`, `confusion_matrix.png` 확인
- 임의의 이미지에 대해 추론 실행

**어그멘테이션 결과 저장**

In [ ]:
from ultralytics import YOLO
import os, glob

# 새로 학습한 run 이름으로 경로를 수정
best_ckpt = "runs_detect/fire_detect/weights/best.pt"
last_ckpt = "runs_detect/fire_detect/weights/last.pt"

ckpt = best_ckpt if os.path.exists(best_ckpt) else last_ckpt
print("사용 가중치:", ckpt)

if os.path.exists(ckpt):
    model = YOLO(ckpt)

    # 추론에 사용할 테스트 이미지 경로 설정 (변경 필요 없음)
    test_imgs = glob.glob(os.path.join("/content/datasets", "**", "valid", "images", "*.jpg"), recursive=True)[:4]
    if len(test_imgs) == 0:
        test_imgs = glob.glob(os.path.join("/content/datasets", "**", "images", "*.jpg"), recursive=True)[:4]

    if test_imgs:
        # save=True 시, 결과는 runs/detect/predictN 폴더에 저장됩니다.
        preds = model.predict(test_imgs, save=True, conf=0.25)
        print("추론 완료. runs/detect/predict 폴더를 확인하세요.")
    else:
        print("테스트 이미지가 없습니다. 추론 스킵.")
else:
    print("체크포인트를 찾지 못했습니다.")

사용 가중치: runs_detect/fire_detect/weights/last.pt
체크포인트를 찾지 못했습니다.


TTA 라이브러리 및 사전 변수 정의





In [ ]:
# TTA를 위한 라이브러리 추가 (Colab 환경에서 설치 필요)
# !pip -q install albumentations

import albumentations as A
import cv2 # cv2.imread, cv2.imwrite 사용

# TTA 변형을 적용하는 함수
def apply_tta_augmentations(image_path, aug_pipeline):
    """주어진 이미지 경로에 TTA 파이프라인을 적용하고 결과를 반환합니다."""
    # OpenCV를 사용하여 이미지 읽기 (albumentations 호환)
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Image not found at {image_path}")

    # BGR을 RGB로 변환 (Albumentations는 RGB를 선호)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # 변형 적용
    augmented = aug_pipeline(image=image)
    aug_image = augmented['image']

    # 다시 BGR로 변환하여 YOLOv8 추론 준비 (YOLOv8 내부에서는 RGB로 처리되지만, cv2 입출력 일관성을 위해)
    aug_image_bgr = cv2.cvtColor(aug_image, cv2.COLOR_RGB2BGR)

    return aug_image_bgr

# TTA 파이프라인 정의 (밝기, 대비, 색상 채도/색조 변형)
TTA_AUGMENTATION = A.Compose([
    # 밝기 및 대비 변형
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
    # 색상 채도/색조 변형
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=1.0)
])

# 재시도에 사용할 Confidence 임계값 리스트
CONFIDENCE_LEVELS = [0.25, 0.20, 0.15, 0.10]

실제 탐지 및 화재 섹터 출력



이미지 한장으로 **앙상블**

In [ ]:
from ultralytics import YOLO
import os, glob
import numpy as np
import math
from collections import defaultdict

# --- 헬퍼 함수: 거리 계산 ---
def euclidean_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

# --- 헬퍼 함수: 섹터 할당 ---
def assign_sector_by_rule(coords_list):
    """정규화된 9개 건물 좌표 리스트를 받아 섹터 번호(1-9)를 매핑합니다."""
    if len(coords_list) != 9:
        return None

    indexed_coords = [(x, y, i) for i, (x, y) in enumerate(coords_list)]
    indexed_coords.sort(key=lambda item: item[0])

    sector_map = {}

    s7_coord = indexed_coords[0]; sector_map[(s7_coord[0], s7_coord[1])] = 7
    s1_coord = indexed_coords[1]; sector_map[(s1_coord[0], s1_coord[1])] = 1

    central_group = indexed_coords[2:5]
    central_group.sort(key=lambda item: item[1])

    s2_coord = central_group[0]; sector_map[(s2_coord[0], s2_coord[1])] = 2
    s6_coord = central_group[1]; sector_map[(s6_coord[0], s6_coord[1])] = 6
    s8_coord = central_group[2]; sector_map[(s8_coord[0], s8_coord[1])] = 8

    remaining_group = indexed_coords[5:9]
    remaining_group.sort(key=lambda item: item[0])

    s3_coord = remaining_group[0]; sector_map[(s3_coord[0], s3_coord[1])] = 3
    s4_coord = remaining_group[-1]; sector_map[(s4_coord[0], s4_coord[1])] = 4

    temp_coords = [coord for coord in remaining_group if coord not in [s3_coord, s4_coord]]
    temp_coords.sort(key=lambda item: item[1])

    s5_coord = temp_coords[0]; sector_map[(s5_coord[0], s5_coord[1])] = 5
    s9_coord = temp_coords[1]; sector_map[(s9_coord[0], s9_coord[1])] = 9

    return sector_map

# --- 헬퍼 함수: 섹터 규칙 검증 ---
def post_verify_sector_rules(sector_map):
    """할당된 섹터 맵에 대해 사후 검증 규칙을 적용합니다."""
    if not sector_map:
        return True, "검증 스킵 (맵 생성 실패)"

    coord_to_sector = {v: k for k, v in sector_map.items()}

    # Y좌표 분리 검증 (섹터 1, 2, 3, 4가 5, 6, 7, 8, 9보다 위에 있어야 함)
    # y좌표는 이미지에서 위에서 아래로 증가하는 경우가 많으므로 '위'는 더 낮은 y값일 수 있음.
    # 여기서는 'top' 그룹의 최대 y값이 'bottom' 그룹의 최소 y값보다 작아야 함을 가정.
    top_y_coords = [coord_to_sector[s][1] for s in [1, 2, 3, 4]]
    bottom_y_coords = [coord_to_sector[s][1] for s in [5, 6, 7, 8, 9]]

    max_top_y = max(top_y_coords)
    min_bottom_y = min(bottom_y_coords)

    if max_top_y >= min_bottom_y:
        return False, f"⚠️ 규칙 1 위반: Y좌표 분리 실패 (상단 Max Y:{max_top_y:.2f} >= 하단 Min Y:{min_bottom_y:.2f})"

    # X좌표 순서 검증 (섹터 1, 2, 3, 4의 X좌표는 순서대로 증가해야 함)
    x_coords = [coord_to_sector[s][0] for s in [1, 2, 3, 4]]
    is_ordered = all(x_coords[i] < x_coords[i+1] for i in range(len(x_coords) - 1))

    if not is_ordered:
        return False, f"⚠️ 규칙 2 위반: X좌표 순서 실패 (섹터 1,2,3,4의 X:{[f'{x:.2f}' for x in x_coords]})"

    return True, "✅ 모든 사후 검증 규칙 만족"

# --- 메인 실행 함수 (단일 이미지 & 다중 Conf 앙상블 로직) ---
def run_fire_detection_and_ensemble_single_image():

    # --- 환경 설정 (이 부분을 실제 경로로 변경해야 합니다) ---
    MODEL_RESULTS_DIR = "/content/drive/MyDrive/공군AI/yolo_results/runs_detect/final"
    TEST_IMAGES_DIR = "/content/drive/MyDrive/공군AI/test/test/images"
    # 사용할 Confidence 레벨 목록 (이 중 3개가 앙상블에 사용됨)
    CONFIDENCE_LEVELS = [0.25, 0.20, 0.15, 0.10, 0.05]

    # --- 모델 로드 ---
    best_ckpt = os.path.join(MODEL_RESULTS_DIR, "weights/best.pt")
    last_ckpt = os.path.join(MODEL_RESULTS_DIR, "weights/last.pt")

    ckpt = best_ckpt if os.path.exists(best_ckpt) else last_ckpt
    print("\n사용 가중치:", ckpt)

    if not os.path.exists(ckpt):
        print("\n🚨 체크포인트 파일 없음.")
        return

    model = YOLO(ckpt)

    test_imgs_all = sorted(glob.glob(os.path.join(TEST_IMAGES_DIR, "*.jpg")))

    if not test_imgs_all:
         print(f"🚨 테스트 이미지가 없습니다. 경로 확인: {TEST_IMAGES_DIR}")
         return

    # 앙상블을 위해 사용할 단일 이미지 선택 (첫 번째 이미지)
    target_image_path = test_imgs_all[0]
    target_image_filename = os.path.basename(target_image_path)
    print(f"🚀 단일 이미지 앙상블 추론 시작: {target_image_filename}")
    print("\n--- 화재 섹터 번호 및 사후 검증 ---")

    best_sector_map = None
    final_fire_buildings = None
    final_validation_msg = None
    final_conf_thresh = None
    W, H = None, None

    # 1. 9개 건물 감지 및 섹터 맵 생성 (Confidence 재시도 루프)
    for attempt, conf_thresh in enumerate(CONFIDENCE_LEVELS):
        print(f"\n--- 시도 {attempt+1} (9개 건물 확보 Conf={conf_thresh:.2f}) ---")

        print(f"  [건물 감지] - 단일 이미지 ({target_image_filename})로 9개 건물 시도...")

        results = model.predict(target_image_path, save=False, conf=conf_thresh, verbose=False)
        pred = results[0]

        # 이미지 크기는 첫 성공 시점에만 저장
        if W is None or H is None:
            W, H = pred.orig_shape[1], pred.orig_shape[0]

        all_building_coords = []
        for box in pred.boxes:
            cls_id = int(box.cls.item())
            class_name = model.names.get(cls_id)
            # 'normal'과 'fire' 모두 건물로 간주
            if class_name in ['normal', 'fire']:
                cx_pix, cy_pix, _, _ = box.xywh[0].tolist()
                norm_x = (cx_pix / W) * 1000
                norm_y = (cy_pix / H) * 1000
                all_building_coords.append((norm_x, norm_y))

        if len(all_building_coords) != 9:
            print(f"    ❌ 9개 건물 감지 실패 ({len(all_building_coords)}개). 다음 Conf로 재시도.")
            continue

        # 2. 섹터 할당 및 사후 검증
        SECTOR_MAPPING_TABLE = assign_sector_by_rule(all_building_coords)
        is_valid, validation_msg = post_verify_sector_rules(SECTOR_MAPPING_TABLE)

        print(f"    ✅ 사후 검증 결과: {validation_msg}")

        if not is_valid:
            print("    ⚠️ 사후 검증 실패. 다음 Conf로 재시도.")
            continue

        # 9개 건물 탐지 및 섹터 규칙 통과 완료.
        best_sector_map = SECTOR_MAPPING_TABLE
        final_validation_msg = validation_msg
        print(f"    ✅ 9개 건물 확정 (Conf={conf_thresh:.2f}). 화재 앙상블 단계로 이동.")
        break

    # ----------------------------------------------------------------
    # 9개 건물 확보 실패 시 종료
    # ----------------------------------------------------------------
    if best_sector_map is None:
        print(f"\n🚨 최종 처리 실패: 모든 Confidence Level에서 9개 건물 및 섹터 맵 확보 실패.")
        return

    # ----------------------------------------------------------------
    # 3. 단일 이미지에 대해 다중 Conf 앙상블 (화재 감지)
    # ----------------------------------------------------------------
    print("\n--- 화재 건물 다중 Conf 앙상블 시작 ---")

    # 앙상블에 사용할 상위 3개의 Confidence Level
    ensemble_conf_levels = CONFIDENCE_LEVELS[:3]
    all_fire_buildings_combined = []

    for conf_level in ensemble_conf_levels:
        print(f"  [Conf={conf_level:.2f} 추론]...")

        # 단일 이미지에 대해 해당 Conf로 추론
        results = model.predict(target_image_path, save=False, conf=conf_level, verbose=False)
        pred = results[0]

        for box in pred.boxes:
            cls_id = int(box.cls.item())
            class_name = model.names.get(cls_id)

            if class_name == 'fire':
                cx_pix, cy_pix, _, _ = box.xywh[0].tolist()
                norm_x = (cx_pix / W) * 1000
                norm_y = (cy_pix / H) * 1000

                all_fire_buildings_combined.append({
                    "center_norm": (norm_x, norm_y),
                    "confidence": box.conf.item(), # 실제 예측 Conf
                    "source_conf": conf_level,     # 추론에 사용된 임계값
                    "source_img": target_image_filename
                })

    print(f"  💡 앙상블에 사용된 Conf 임계값: {', '.join([f'{c:.2f}' for c in ensemble_conf_levels])}")
    print(f"  💡 총 화재 후보 개수: {len(all_fire_buildings_combined)}개")


    # 4. 최종 2개 선택 (신뢰도 순으로 정렬 후 상위 2개)
    all_fire_buildings_combined.sort(key=lambda x: x['confidence'], reverse=True)
    fire_to_process = all_fire_buildings_combined[:2]

    # 5. 최종 결과 저장 (2개 미만이라도 최종 결과로 사용)
    final_fire_buildings = fire_to_process
    final_conf_thresh = ensemble_conf_levels[0] # 가장 높은 Conf를 대표로 표시

    # ----------------------------------------------------------------
    # Unique Sector 출력 로직 (중복 섹터 제거 및 최고 Conf 선택)
    # ----------------------------------------------------------------
    if not final_fire_buildings:
        print("\n🚨 최종 처리 실패: 앙상블 후 화재 건물 감지 실패.")
        print("==================================================")
        return

    # Step 1: Map all confirmed fire candidates (up to 2) to sectors
    raw_fire_sectors = []
    for fire_data in final_fire_buildings:
        fire_coord_norm = fire_data["center_norm"]
        fire_conf = fire_data["confidence"]

        min_dist = float('inf')
        closest_sector = None

        for sector_coord, sector_num in best_sector_map.items():
            distance = euclidean_distance(fire_coord_norm, sector_coord)
            if distance < min_dist:
                min_dist = distance
                closest_sector = sector_num

        raw_fire_sectors.append({
            "sector": closest_sector,
            "confidence": fire_conf,
            "source_img": fire_data.get("source_img", "N/A"),
            "source_conf_used": fire_data.get("source_conf", "N/A")
        })

    # Step 2: Group by sector and keep the highest confidence for each unique sector
    unique_fire_sectors_map = {}
    for s_data in raw_fire_sectors:
        sector_num = s_data['sector']
        current_conf = s_data['confidence']

        # 이미 존재하는 섹터가 있으면, 더 높은 Conf로 업데이트 (중복 섹터 제거)
        if sector_num not in unique_fire_sectors_map or current_conf > unique_fire_sectors_map[sector_num]['confidence']:
            unique_fire_sectors_map[sector_num] = s_data

    # Step 3: Final list and sorting
    unique_fire_sectors = list(unique_fire_sectors_map.values())
    unique_fire_sectors.sort(key=lambda x: x['sector'])

    # 최종 출력 구성
    fire_count = len(unique_fire_sectors)

    output_list = []
    for s in unique_fire_sectors:
        # 이 섹터에 기여한 모든 앙상블 출처를 모읍니다.
        # 이 경우 'source_conf_used'를 포함하여 어떤 Conf에서 감지되었는지 보여줍니다.
        all_sources_conf = [
            f"Conf {data['source_conf_used']:.2f}" for data in raw_fire_sectors
            if data['sector'] == s['sector']
        ]
        # 중복 제거 및 정렬
        source_str = ", ".join(sorted(list(set(all_sources_conf))))

        output_list.append(
            f"Sector {s['sector']} (최고 신뢰도: {s['confidence']:.2f}, 기여 임계값: {source_str})"
        )

    # 최종 출력
    print("\n==================================================")
    print(f"✅ 최종 감지 성공 (9개 건물 Conf:{final_conf_thresh:.2f}, {final_validation_msg})")
    print(f"🔥 확정 화재 건물 개수: **{fire_count}개**")
    print(f"🔥 화재 건물 목록: **{', '.join(output_list)}**")
    print(f"💡 단일 이미지 출처: {target_image_filename}")
    print("==================================================")

# --- 실행부 (Colab 환경에서 사용 시 아래 주석을 해제) ---
run_fire_detection_and_ensemble_single_image()


사용 가중치: /content/drive/MyDrive/공군AI/yolo_results/runs_detect/final/weights/best.pt
🚀 단일 이미지 앙상블 추론 시작: img_sector1_00_png.rf.4ba75c2abbedf619c25ef85eb9117d41.jpg

--- 화재 섹터 번호 및 사후 검증 ---

--- 시도 1 (9개 건물 확보 Conf=0.25) ---
  [건물 감지] - 단일 이미지 (img_sector1_00_png.rf.4ba75c2abbedf619c25ef85eb9117d41.jpg)로 9개 건물 시도...
    ✅ 사후 검증 결과: ✅ 모든 사후 검증 규칙 만족
    ✅ 9개 건물 확정 (Conf=0.25). 화재 앙상블 단계로 이동.

--- 화재 건물 다중 Conf 앙상블 시작 ---
  [Conf=0.25 추론]...
  [Conf=0.20 추론]...
  [Conf=0.15 추론]...
  💡 앙상블에 사용된 Conf 임계값: 0.25, 0.20, 0.15
  💡 총 화재 후보 개수: 3개

✅ 최종 감지 성공 (9개 건물 Conf:0.25, ✅ 모든 사후 검증 규칙 만족)
🔥 확정 화재 건물 개수: **1개**
🔥 화재 건물 목록: **Sector 1 (최고 신뢰도: 0.85, 기여 임계값: Conf 0.20, Conf 0.25)**
💡 단일 이미지 출처: img_sector1_00_png.rf.4ba75c2abbedf619c25ef85eb9117d41.jpg


In [ ]:
## 🖼️ 섹터 시각화 준비 (OpenCV 및 헬퍼 함수)

import cv2
import os
import glob
import numpy as np

# 시각화 결과를 저장할 디렉토리 경로 (이름을 run 이름과 일치시키세요)
MODEL_RESULTS_DIR = "/content/drive/MyDrive/공군AI/yolo_results/runs_detect/final"
VISUALIZATION_OUTPUT_DIR = os.path.join(os.path.dirname(MODEL_RESULTS_DIR), "sector_viz")
os.makedirs(VISUALIZATION_OUTPUT_DIR, exist_ok=True)

print(f"시각화 결과 저장 경로: {VISUALIZATION_OUTPUT_DIR}")
print("준비 완료.")

시각화 결과 저장 경로: /content/drive/MyDrive/공군AI/yolo_results/runs_detect/sector_viz
준비 완료.


**시각화**

In [ ]:
import os
import glob
import math
import cv2
from ultralytics import YOLO
import numpy as np
from collections import defaultdict

MODEL_RESULTS_DIR = "/content/drive/MyDrive/공군AI/yolo_results/runs_detect/final"
TEST_IMAGES_DIR = "/content/drive/MyDrive/공군AI/test/test/images"
CONF_THRESH_TO_USE = 0.25
VISUALIZATION_OUTPUT_DIR = os.path.join(os.path.dirname(MODEL_RESULTS_DIR), "sector_viz")


def euclidean_distance(p1, p2):
    """유클리드 거리 계산"""
    return ((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)**0.5

def assign_sector_by_rule(coords_list):
    """정규화된 9개 건물 좌표 리스트를 받아 섹터 번호(1-9)를 매핑합니다."""
    if len(coords_list) != 9:
        return None

    indexed_coords = [(x, y, i) for i, (x, y) in enumerate(coords_list)]
    indexed_coords.sort(key=lambda item: item[0])

    sector_map = {}

    s7_coord = indexed_coords[0]; sector_map[(s7_coord[0], s7_coord[1])] = 7
    s1_coord = indexed_coords[1]; sector_map[(s1_coord[0], s1_coord[1])] = 1

    central_group = indexed_coords[2:5]
    central_group.sort(key=lambda item: item[1])

    s2_coord = central_group[0]; sector_map[(s2_coord[0], s2_coord[1])] = 2
    s6_coord = central_group[1]; sector_map[(s6_coord[0], s6_coord[1])] = 6
    s8_coord = central_group[2]; sector_map[(s8_coord[0], s8_coord[1])] = 8

    remaining_group = indexed_coords[5:9]
    remaining_group.sort(key=lambda item: item[0])

    s3_coord = remaining_group[0]; sector_map[(s3_coord[0], s3_coord[1])] = 3
    s4_coord = remaining_group[-1]; sector_map[(s4_coord[0], s4_coord[1])] = 4

    temp_coords = [coord for coord in remaining_group if coord not in [s3_coord, s4_coord]]
    temp_coords.sort(key=lambda item: item[1])

    s5_coord = temp_coords[0]; sector_map[(s5_coord[0], s5_coord[1])] = 5
    s9_coord = temp_coords[1]; sector_map[(s9_coord[0], s9_coord[1])] = 9

    return sector_map
# --- 헬퍼 함수 끝 ---


def visualize_sector_map():
    """YOLO 탐지 및 섹터 매핑 결과를 시각화하고 저장합니다."""

    # 결과 디렉토리 생성
    os.makedirs(VISUALIZATION_OUTPUT_DIR, exist_ok=True)
    print(f"시각화 결과 저장 경로: {VISUALIZATION_OUTPUT_DIR}")

    # 모델 로드
    best_ckpt = os.path.join(MODEL_RESULTS_DIR, "weights/best.pt")
    if not os.path.exists(best_ckpt):
        print(f"🚨 체크포인트 파일 없음: {best_ckpt}. 경로를 확인하세요.")
        return

    model = YOLO(best_ckpt)

    # 테스트 이미지 선택
    test_imgs_all = sorted(glob.glob(os.path.join(TEST_IMAGES_DIR, "*.jpg")))
    if not test_imgs_all:
        print(f"🚨 테스트 이미지가 없습니다. 경로를 확인하세요: {TEST_IMAGES_DIR}")
        return

    first_image_path = test_imgs_all[0]
    print(f"\n사용할 이미지: {os.path.basename(first_image_path)}")

    # --- 1. 9개 건물 감지 및 데이터 저장 ---
    print(f"[{CONF_THRESH_TO_USE:.2f}] Confidence로 9개 건물 감지 시도...")
    results_first = model.predict(first_image_path, save=False, conf=CONF_THRESH_TO_USE, verbose=False)
    pred_first = results_first[0]
    W, H = pred_first.orig_shape[1], pred_first.orig_shape[0] # 이미지 폭과 높이

    all_building_data = []
    for box in pred_first.boxes:
        cls_id = int(box.cls.item())
        class_name = model.names.get(cls_id)

        if class_name in ['normal', 'fire']:
            # 정규화된 중심 좌표 (섹터 매핑에 사용)
            cx_pix, cy_pix, _, _ = box.xywh[0].tolist()
            norm_x = (cx_pix / W) * 1000
            norm_y = (cy_pix / H) * 1000

            # 바운딩 박스 픽셀 좌표 [x1, y1, x2, y2]
            xyxy_pix = [int(x) for x in box.xyxy[0].tolist()]

            all_building_data.append({
                "norm_center": (norm_x, norm_y),
                "xyxy_pix": xyxy_pix,
                "class_name": class_name,
                "conf": box.conf.item(), # 데이터는 유지하되, 시각화 시 제외
            })

    # 섹터 매핑을 위한 데이터 준비
    NORM_CENTER_TO_DATA = {data["norm_center"]: data for data in all_building_data}
    all_building_coords_first = [data["norm_center"] for data in all_building_data]

    if len(all_building_coords_first) != 9:
        print(f"🚨 9개 건물 감지 실패 ({len(all_building_coords_first)}개). 시각화 스킵.")
        return

    SECTOR_MAPPING_TABLE = assign_sector_by_rule(all_building_coords_first)

    # --- 2. 섹터 시각화 실행 ---
    print("\n🖼️ 시각화 및 저장 시작 (가독성 개선 적용, 신뢰도 제외)...")

    # 이미지 로드 (OpenCV는 BGR 순서)
    img_to_visualize = cv2.imread(first_image_path)

    if img_to_visualize is None:
        print(f"❌ 시각화를 위한 이미지 로드 실패: {first_image_path}")
        return

    # 텍스트 스타일 개선
    LABEL_FONT_FACE = cv2.FONT_HERSHEY_DUPLEX
    LABEL_FONT_SCALE = 0.6
    LABEL_FONT_THICKNESS = 1
    LABEL_TEXT_COLOR = (0, 0, 0)         # BGR: 흰색

    # 각 섹터에 대해 시각화
    for norm_coord, sector_num in SECTOR_MAPPING_TABLE.items():
        building_data = NORM_CENTER_TO_DATA.get(norm_coord)
        if not building_data:
            continue

        x1, y1, x2, y2 = building_data["xyxy_pix"]
        class_name = building_data["class_name"]

        # 1. 바운딩 박스 그리기 (클래스별 색상 지정)
        if class_name == 'fire':
            color = (0, 0, 255)    # BGR: Red (화재)
        else:
            color = (0, 240, 0)    # BGR: Green (정상)

        thickness = 3 # 바운딩 박스 굵기
        cv2.rectangle(img_to_visualize, (x1, y1), (x2, y2), color, thickness)

        # 2. 텍스트 표시 (바운딩 박스 상단 '밖으로' 이동)
        label = f"S{sector_num}, {class_name}"

        # 텍스트 크기 계산
        (lw, lh), bl = cv2.getTextSize(label, LABEL_FONT_FACE, LABEL_FONT_SCALE, LABEL_FONT_THICKNESS)

        # 텍스트 위치 계산 (바운딩 박스 상단에서 위로 10픽셀 띄우기)
        text_x_start = x1
        text_y_start = y1 - 5

        # 텍스트 배경 (Class Label)
        # 배경 상자를 텍스트 시작 위치에 맞게 그림
        cv2.rectangle(img_to_visualize,
                      (text_x_start, text_y_start - lh - 2),
                      (text_x_start + lw + 2, text_y_start + 2),
                      color, -1) # 배경색

        # 텍스트 (흰색)
        cv2.putText(img_to_visualize, label,
                    (text_x_start + 1, text_y_start - 1),
                    LABEL_FONT_FACE, LABEL_FONT_SCALE, LABEL_TEXT_COLOR, LABEL_FONT_THICKNESS)


    # --- 3. 파일 저장 ---
    vis_filename = f"sector_map_conf{CONF_THRESH_TO_USE:.2f}_yolo_viz_clean_{os.path.basename(first_image_path)}"
    vis_output_path = os.path.join(VISUALIZATION_OUTPUT_DIR, vis_filename)

    cv2.imwrite(vis_output_path, img_to_visualize)
    print(f"✅ 시각화 결과 저장 완료: {vis_output_path}")

if __name__ == "__main__":
    visualize_sector_map()

시각화 결과 저장 경로: /content/drive/MyDrive/공군AI/yolo_results/runs_detect/sector_viz

사용할 이미지: img_sector1_00_png.rf.4ba75c2abbedf619c25ef85eb9117d41.jpg
[0.25] Confidence로 9개 건물 감지 시도...

🖼️ 시각화 및 저장 시작 (가독성 개선 적용, 신뢰도 제외)...
✅ 시각화 결과 저장 완료: /content/drive/MyDrive/공군AI/yolo_results/runs_detect/sector_viz/sector_map_conf0.25_yolo_viz_clean_img_sector1_00_png.rf.4ba75c2abbedf619c25ef85eb9117d41.jpg


In [ ]:
## 💾 Google Drive에 시각화 결과 복사

USE_DRIVE = True
if USE_DRIVE:
    # 이미 마운트되어 있다면 이 코드는 스킵됩니다.
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')

    # 시각화 결과 폴더를 드라이브에 복사
    DRIVE_VIS_PATH = os.path.join(os.path.dirname(MODEL_RESULTS_DIR), "sector_viz")
    !cp -r "$DRIVE_VIS_PATH" /content/drive/MyDrive/yolo_results/
    print(f"\n✅ 시각화 폴더 '{os.path.basename(DRIVE_VIS_PATH)}'가 Google Drive에 복사되었습니다.")

cp: cannot stat '/content/drive/MyDrive/공군AI/yolo_results/runs_detect/sector_viz': No such file or directory

✅ 시각화 폴더 'sector_viz'가 Google Drive에 복사되었습니다.


## Google Drive에 결과 저장
Colab에서 학습 결과를 유지하려면 Google Drive를 마운트해서 결과 폴더를 복사하세요.

Google Drive에 백업 하지 않은 채 Colab 창을 닫으면 파일이 날아갑니다!!

best.pt를 꼭 로컬에 저장 혹은 구글 드라이브에 업로드 하세요.

In [ ]:
USE_DRIVE = True  # True로 바꾸면 드라이브에 저장
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/yolo_results
    !cp -r runs_detect /content/drive/MyDrive/yolo_results/

Mounted at /content/drive
cp: cannot stat 'runs_detect': No such file or directory
